### Display MTEB results
MTEB results are saved in .json files, one for each task. 
This notebook aggregates the results for multiple tasks and multiple models.

In [ ]:
import os
os.getcwd()

In [ ]:
import mteb
import json
import pandas as pd
import numpy as np

In [3]:
# get task type 
mteb.get_task("ArguAna").metadata.type

'Retrieval'

#### Analyze MTEB results of different TF-IDF configurations

In [ ]:
data = {}
main_dir = "text_embedding/MTEB/sparse_results"

# Traverse folders and subfolders
for (root, dirs, files) in os.walk(main_dir):
    # Identify model version from the folder structure
    model_version = os.path.basename(root)
    print(model_version)
        
    for file in files:
        # Skip unwanted files
        if file in {"model_meta.json"} or not file.endswith('.json'):
            continue
            
        file_path = os.path.join(root, file)
        try:
            # Read JSON and extract necessary fields
            with open(file_path, 'r') as f:
                json_data = json.load(f)
                task_name = json_data.get("task_name")
                main_score = json_data.get("scores", {}).get("test", {})[0]["main_score"]
                main_score = round(main_score*100, 2)
                    
                if task_name and main_score is not None:
                    if task_name not in data:
                        data[task_name] = {}
                    data[task_name][model_version] = main_score
        except (json.JSONDecodeError, KeyError):
            print(f"Error parsing file: {file_path}")


In [ ]:
df = pd.DataFrame.from_dict(data, orient='index')
df.index.name = "task_name"
df["task_types"] = [mteb.get_task(task).metadata.type for task in df.index]
df

In [ ]:
task_selection = ["ArguAna", "ArxivClusteringP2P", "BiorxivClusteringP2P", "MedrxivClusteringP2P", "MindSmallReranking",
                 "RedditClusteringP2P", "SCIDOCS", "SciDocsRR", "StackExchangeClusteringP2P", "STS15", "STS16",
                 "STSBenchmark"]

df.loc[task_selection].sort_values("task_types")

In [7]:
df[["svd50_log", "svd_log_run2", "svd200_log", "svd300_log", "svd500_log", "task_types"]].loc[task_selection].sort_values("task_types")

,svd50_log,svd_log_run2,svd200_log,svd300_log,svd500_log,task_types
task_name,,,,,,
ArxivClusteringP2P,39.75,40.23,40.36,40.60,40.84,Clustering
BiorxivClusteringP2P,33.62,33.74,33.28,33.64,33.26,Clustering
MedrxivClusteringP2P,29.81,30.11,29.97,29.87,29.39,Clustering
RedditClusteringP2P,33.98,34.59,36.51,38.23,40.91,Clustering
StackExchangeClusteringP2P,36.00,34.12,32.62,31.48,29.73,Clustering
MindSmallReranking,25.75,26.63,27.04,27.31,NaN,Reranking
SciDocsRR,48.32,52.71,56.49,58.65,60.46,Reranking
ArguAna,32.53,41.35,48.24,51.14,54.25,Retrieval
SCIDOCS,4.08,5.31,6.77,7.72,9.61,Retrieval


In [8]:
df[["tfidf_log_run2", "svd_log_old_run1", "svd_log_run2", "svd_log_piecewise", "task_types"]].loc[task_selection].sort_values("task_types")

,tfidf_log_run2,svd_log_old_run1,svd_log_run2,svd_log_piecewise,task_types
task_name,,,,,
ArxivClusteringP2P,34.79,41.73,40.23,40.23,Clustering
BiorxivClusteringP2P,26.21,33.80,33.74,33.74,Clustering
MedrxivClusteringP2P,22.02,30.00,30.11,30.11,Clustering
RedditClusteringP2P,39.94,45.96,34.59,34.59,Clustering
StackExchangeClusteringP2P,17.40,34.01,34.12,34.11,Clustering
MindSmallReranking,22.49,NaN,26.63,NaN,Reranking
SciDocsRR,62.32,NaN,52.71,NaN,Reranking
ArguAna,52.52,NaN,41.35,NaN,Retrieval
SCIDOCS,14.73,NaN,5.31,NaN,Retrieval


#### Analyze kNN results of different TF-IDF configurations

In [11]:
data = {}
main_dir = "../MTEB/knn_results"

# Traverse folders and subfolders
for (root, dirs, files) in os.walk(main_dir):
        
    for file in files:
        # Skip unwanted files
        model_name = file.strip(".json")
            
        file_path = os.path.join(root, file)
        try:
            # Read JSON and extract necessary fields
            with open(file_path, 'r') as f:
                json_data = json.load(f)
                for key, value in json_data.items():
                    if type(value) == list:
                        value = np.mean(value)
                    json_data[key] = round(value*100, 2)

                data[model_name] = json_data
        except (json.JSONDecodeError, KeyError):
            print(f"Error parsing file: {file_path}")


In [12]:
pd.DataFrame.from_dict(data)

,Tfidf_svd_log_old,Tfidf_svd300_log,Tfidf_svd_log,Tfidf_svd_log_old_novocab_run1,Tfidf_svd_log_old_novocab,Tfidf_svd50_log,Tfidf_svd200_log
arxiv_full,40.41,41.86,40.41,NaN,40.41,39.05,41.21
biorxiv_full,62.54,62.82,62.54,62.54,62.54,62.82,63.06
medrxiv_full,61.63,63.97,61.63,61.63,61.63,60.48,64.59
reddit_full,49.65,51.97,49.65,49.65,49.65,48.16,52.15
stackexchange_full,46.88,47.63,46.88,46.88,46.88,46.25,47.99
arxiv_batchwise,66.61,66.15,64.58,66.61,66.61,63.03,65.60
medrxiv_batchwise,49.40,49.80,49.06,49.40,49.40,48.04,49.60
reddit_batchwise,78.81,69.64,69.36,78.81,78.81,68.53,70.56
stackexchange_batchwise,39.16,38.81,38.23,39.16,39.16,38.17,38.62
